# MHAR Benchmark Model Comparison


**Purpose:** This notebook compares the out-of-sample test-set performance of the MHAR benchmark models `HAR`, `LogHAR`, `LevHAR`, `SHAR`, and `HARQ`. Following the presentation in the reference paper, we construct a pairwise relative-MSE table and complement it with one-sided Diebold-Mariano (DM) tests for equal predictive accuracy. All comparisons are based on the saved one-day-ahead realized-variance forecasts from the benchmark-model notebooks.


## 1. Load Saved Forecasts


In [ ]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from IPython.display import display

forecast_path = "/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/Cleaned Data/Forcasts/MHAR"
models = ["HAR", "LogHAR", "LevHAR", "SHAR", "HARQ"]

forecast_files = {
    "HAR": "forecasts_HAR.csv",
    "LogHAR": "forecasts_LogHAR.csv",
    "LevHAR": "forecasts_LevHAR.csv",
    "SHAR": "forecasts_SHAR.csv",
    "HARQ": "forecasts_HARQ.csv",
}

results = {}
for model in models:
    df = pd.read_csv(
        os.path.join(forecast_path, forecast_files[model]),
        index_col=0,
        parse_dates=True
    )
    results[model] = df

for model, df in results.items():
    print(f"{model:<7} | shape={df.shape} | period: {df.index.min().date()} to {df.index.max().date()}")


## 2. Align Forecast Series


The saved benchmark forecasts should already share the same test period. We align them on the common index before computing error metrics and pairwise comparisons.


In [ ]:
common_index = results["HAR"].index
for model in models[1:]:
    common_index = common_index.intersection(results[model].index)

for model in models:
    results[model] = results[model].loc[common_index].copy()

print(f"Common test observations: {len(common_index)}")
print(f"Common period: {common_index[0].date()} to {common_index[-1].date()}")

actual_reference = results["HAR"]["RV_actual"]
consistency_check = pd.DataFrame({
    model: np.allclose(results[model]["RV_actual"].values, actual_reference.values)
    for model in models
}, index=["Actuals match HAR"])
display(consistency_check)


## 3. Test-Set Summary Metrics


In [ ]:
summary_rows = []
for model in models:
    df = results[model]
    errors = df["RV_actual"] - df["RV_forecast"]
    mse = np.mean(errors**2)
    summary_rows.append({
        "Model": model,
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "MAE": np.mean(np.abs(errors)),
        "Mean Error": np.mean(errors),
    })

summary_metrics = pd.DataFrame(summary_rows).set_index("Model").sort_values("MSE")
summary_metrics["Relative MSE vs HAR"] = summary_metrics["MSE"] / summary_metrics.loc["HAR", "MSE"]

display(
    summary_metrics.style.format({
        "MSE": "{:.6e}",
        "RMSE": "{:.6e}",
        "MAE": "{:.6e}",
        "Mean Error": "{:.6e}",
        "Relative MSE vs HAR": "{:.4f}",
    }).set_caption(
        "Table 1: Test-Set Forecast Performance Summary — MHAR Benchmark Models"
    )
)


## 4. Diebold-Mariano Test Setup


We use a one-sided Diebold-Mariano test with squared-error loss. For each row-model $i$ and column-model $j$, the null is equal predictive accuracy, while the alternative is that model $j$ has lower expected squared forecast error than model $i$. With one-step-ahead forecasts, the HAC bandwidth is zero in practice, but we keep the same Newey-West-style implementation as in the repo's forecast-evaluation code.


In [ ]:
def diebold_mariano_test(e1: np.ndarray, e2: np.ndarray, h: int = 1) -> tuple[float, float]:
    '''
    H0: E[d_t] = 0, where d_t = e1_t^2 - e2_t^2
    H1: E[d_t] > 0, meaning model 2 beats model 1.
    Returns: (DM statistic, one-sided p-value)
    '''
    d = e1**2 - e2**2
    d_bar = d.mean()
    T = len(d)

    nw = sm.stats.sandwich_covariance.cov_hac_simple(
        sm.OLS(d, np.ones(T)).fit(), nlags=max(1, h - 1)
    )
    se = np.sqrt(nw[0, 0] / T)

    dm_stat = d_bar / se
    p_val = 1 - stats.norm.cdf(dm_stat)
    return float(dm_stat), float(p_val)


def significance_stars(p_value: float) -> str:
    if p_value < 0.01:
        return "***"
    if p_value < 0.05:
        return "**"
    if p_value < 0.10:
        return "*"
    return ""


## 5. Paper-Style Relative MSE and DM Table


Each cell reports the test-set MSE of the model in the selected column relative to the benchmark in the selected row. A value below 1 means the column model outperforms the row benchmark. Significance stars refer to the one-sided DM test in favor of the column model: `*` for 10%, `**` for 5%, and `***` for 1%.


In [ ]:
relative_mse_numeric = pd.DataFrame(index=models, columns=models, dtype=float)
dm_stat_matrix = pd.DataFrame(index=models, columns=models, dtype=float)
dm_pval_matrix = pd.DataFrame(index=models, columns=models, dtype=float)
paper_style_table = pd.DataFrame(index=models, columns=models, dtype=object)

for row_model in models:
    e_row = results[row_model]["RV_actual"].values - results[row_model]["RV_forecast"].values
    mse_row = np.mean(e_row**2)
    for col_model in models:
        if row_model == col_model:
            relative_mse_numeric.loc[row_model, col_model] = np.nan
            dm_stat_matrix.loc[row_model, col_model] = np.nan
            dm_pval_matrix.loc[row_model, col_model] = np.nan
            paper_style_table.loc[row_model, col_model] = "–"
            continue

        e_col = results[col_model]["RV_actual"].values - results[col_model]["RV_forecast"].values
        mse_col = np.mean(e_col**2)
        ratio = mse_col / mse_row
        dm_stat, dm_pval = diebold_mariano_test(e_row, e_col, h=1)
        stars = significance_stars(dm_pval) if ratio < 1 else ""

        relative_mse_numeric.loc[row_model, col_model] = ratio
        dm_stat_matrix.loc[row_model, col_model] = dm_stat
        dm_pval_matrix.loc[row_model, col_model] = dm_pval
        paper_style_table.loc[row_model, col_model] = f"{ratio:.3f}{stars}"

display(
    paper_style_table.style.set_caption(
        "Table 2: One-Day-Ahead Relative MSE and Diebold-Mariano Test — MHAR Benchmark Models"
    )
)


## 6. Underlying Numeric Tables


In [ ]:
display(
    relative_mse_numeric.style.format("{:.6f}").set_caption(
        "Table 3: Pairwise Relative MSE Matrix (Column Model / Row Benchmark)"
    )
)

display(
    dm_stat_matrix.style.format("{:.3f}").set_caption(
        "Table 4: Pairwise Diebold-Mariano Statistics (Alternative: Column Model Better Than Row Model)"
    )
)

display(
    dm_pval_matrix.style.format("{:.4f}").set_caption(
        "Table 5: Pairwise One-Sided Diebold-Mariano p-values"
    )
)


## 7. Model Ranking and Interpretation Aid


In [ ]:
ranking = summary_metrics[["MSE", "RMSE", "Relative MSE vs HAR"]].copy()
ranking["Rank by MSE"] = np.arange(1, len(ranking) + 1)
ranking = ranking[["Rank by MSE", "MSE", "RMSE", "Relative MSE vs HAR"]]

print("Ranking by test-set MSE:")
print(ranking.to_string(formatters={
    "MSE": "{:.6e}".format,
    "RMSE": "{:.6e}".format,
    "Relative MSE vs HAR": "{:.4f}".format,
}))

print("\nInterpretation guide:")
print("- Relative MSE < 1: column model beats row benchmark.")
print("- Positive DM statistic: evidence in favor of the column model under squared-error loss.")
print("- Stars in Table 2 indicate one-sided DM significance in favor of the column model.")
